**Homework 3 - Caitlin Kontaridis**

Importing...

In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import(
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, log_loss, matthews_corrcoef, balanced_accuracy_score
)
from sklearn.preprocessing import LabelEncoder
from pyspark.sql.functions import col,floor,lit, datediff,current_date,avg, when, min, max, year, sum, variance, expr
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import rank
from pyspark.sql.types import IntegerType, DoubleType
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay

In [0]:
df_laptimes = spark.read.csv("/Volumes/gr5069/raw/f1_data/lap_times.csv", header=True)
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)
pit_stops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)
races = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)
drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)
results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
w_finish = spark.read.csv("/Volumes/gr5069/raw/f1_data/2022_driver_standings.csv", header=True)
w_finish = Window.partitionBy("raceId").orderBy("positionOrder")
w_pit = Window.partitionBy("raceId").orderBy("avg_duration")


In [0]:
results_data = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
display(results_spark)

In [0]:
races_data = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)
display(races_spark)

In [0]:
drivers_data = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)
display(drivers_spark)

In [0]:
print(type(results_data), (results_data.count(), len(results_data.columns)))
print(type(races_data), (races_data.count(), len(races_data.columns)))
print(type(drivers_data), (drivers_data.count(), len(drivers_data.columns)))

Convert to Pandas

In [0]:
results_pd = results_data.pandas_api().to_pandas()
races_pd = races_data.pandas_api().to_pandas()
drivers_pd = drivers_data.pandas_api().to_pandas()
print(type(results_pd), results_pd.shape)
print(type(races_pd), races_pd.shape)
print(type(drivers_pd), drivers_pd.shape)

Merge

In [0]:
merged = results.toPandas().copy()
merged["positionOrder"] = pd.to_numeric(merged["positionOrder"], errors="coerce")
merged["top5"] = (merged["positionOrder"] <= 5).astype(int)
print(merged["top5"].value_counts())

In [0]:
merged = merged.merge(races_pd[["raceId", "year", "round", "circuitId"]], on="raceId", how="left")
merged = merged.merge(drivers_pd[["driverId", "nationality"]], on="driverId", how="left")
print(merged.columns.tolist())

In [0]:
le = LabelEncoder()
merged["nationality"] = merged["nationality"].fillna("Unknown")
merged["nationality"] = le.fit_transform(merged["nationality"].astype(str))
print(merged["nationality"].head())

In [0]:
for col in ["grid", "year", "round", "circuitId", "constructorId", "driverId", "nationality"]:
    merged[col] = pd.to_numeric(merged[col], errors="coerce").fillna(0)
FEATURES = ["grid", "year", "round", "circuitId", "nationality", "constructorId"]
clean = merged[FEATURES + ["top5"]].dropna()
X = clean[FEATURES]
Y = clean["top5"]

X_TRAIN, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
print(X_TRAIN.shape, X_test.shape, Y.mean())

Train First Model

In [0]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)
model.fit(X_TRAIN, y_train)
preds = model.predict(X_test)

In [0]:
#Evaluate
acc = accuracy_score(y_test, preds)
prec = precision_score(y_test, preds)
rec = recall_score(y_test, preds)
print(acc, prec, rec)

MLflow Tracking

In [0]:
with mlflow.start_run():
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=42
    )
    model.fit(X_TRAIN, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 5)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("recall", rec)
    mlflow.sklearn.log_model(model, "model")

Add Artifacts

In [0]:
ConfusionMatrixDisplay.from_predictions(y_test, preds)
plt.savefig("confusion_matrix.png")
mlflow.log_artifact("confusion_matrix.png")
plt.close()

Predictions CSV

In [0]:
results = pd.DataFrame({
    "actual": y_test,
    "predicted": preds
})
results.to_csv("predictions.csv", index=False)
mlflow.log_artifact("predictions.csv")

More Experiments...

In [0]:
mlflow.end_run()
for n in [50, 100, 200]:
    for depth in [3, 5, 10, None]:
        with mlflow.start_run():
            model = RandomForestClassifier(
                n_estimators=n,
                max_depth=depth,
                random_state=42
            )
            model.fit(X_TRAIN, y_train)
            preds = model.predict(X_test)
            acc = accuracy_score(y_test, preds)
            prec = precision_score(y_test, preds)
            rec = recall_score(y_test, preds)
            
            mlflow.log_param("n_estimators", n)
            mlflow.log_param("max_depth", depth)
            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("precision", prec)
            mlflow.log_metric("recall", rec)
            mlflow.sklearn.log_model(model, "model")

Analysis

  Run max_depth n_estimators Accuracy Precision Recall
      suave-cat-181        10          200   0.8397    0.6553 0.4847
  hilarious-fox-783        10          100   0.8397    0.6565 0.4820
      upset-ant-206        10           50   0.8389    0.6514 0.4865
     angry-crab-201         5           50   0.8365    0.6617 0.4390
 thoughtful-fox-231         5          100   0.8363    0.6715 0.4183
       able-pig-188         5          100   0.8363    0.6715 0.4183
   amazing-bass-938         5          200   0.8356    0.6686 0.4165
      rare-wren-684      None          200   0.8309    0.6173 0.4937
     sincere-ox-197      None           50   0.8283    0.6109 0.4820
monumental-fawn-681      None          100   0.8274    0.6070 0.4838
  abrasive-hare-337         3           50   0.8150    0.6782 0.2118
   melodic-slug-512         3          200   0.8128    0.6972 0.1777
  marvelous-fox-952         3          100   0.8102    0.6842 0.1634


The best run was: hilarious-fox-783 (max_depth=10, n_estimators=100)
This is the tie for top accuracy, has slightly better precision, and uses half the trees of suave-cat-181 which makes it more efficient.

Submit

In [0]:
!mkdir -p /tmp/gitconfig
!git config --global --add safe.directory /Workspace/Users/cmk2240@columbia.edu/testrepo-2026/src/HW3_Kontaridis

In [0]:
import os
os.environ["GIT_CONFIG_GLOBAL"] = "/tmp/gitconfig/.gitconfig"

In [0]:
!git config --global --add safe.directory "*"

In [0]:
!git config --global user.email "cmk2240@columbia.edu"
!git config --global user.name "Caitlin Kontaridis"

In [0]:
pwd

In [0]:
%sh
cd /Workspace/Users/cmk2240@columbia.edu/testrepo-2026/src/HW3_Kontaridis
git init

In [0]:
%sh
git remote add origin https://github.com/QMSS-GR5069-Spring2026/take-home-exercise-3-cmk2240-3

In [0]:
%sh
git add .

In [0]:
%sh
git commit -m "HW submission of MLflow model and screenshots"

In [0]:
%sh
git branch -M main

In [0]:
%sh
git status

In [0]:
%sh
git push -u origin main